# Lesson 3: The loop

**Project:** Agentic Engineering Worklog
**Model:** `qwen3:8b`, running on this PC through Ollama

## Where lesson 2 left off

Lesson 2 gave the model one tool, `add_entry`, and scored 6 of 6. But the program gave the model exactly one chance to ask for tools. That works for "log this", and it breaks on a request like:

> Log an hour on Herd Planner updating the calving chart, then tell me how many hours I've logged on Herd Planner today.

That needs two tools, one after the other, and the second depends on the first: the total is only right after the new entry is saved.

## What we are looking for

By the end of this notebook you should be able to say yes to five things:

1. I have a second tool, `search_entries`, that reads the log back.
2. I have seen the lesson 2 program fail on a two-step request.
3. I can write the loop: call the model, run what it asks for, send the results back, and repeat until it stops asking.
4. I know why the loop needs a step limit.
5. I can score the agent on a test set that includes multi-step requests.

This is level 3 from the "plain code, a model call, or an agent" ladder. The model now decides what happens next, so this is the first real agent.

**How to run it in PyCharm:** click a code cell and press **Shift + Enter**. Run the cells in order from top to bottom.

## Step 1: Setup

The same setup as lesson 2, plus today's date, which the model needs to understand "today" and "yesterday".

In [1]:
import json
import re
from datetime import date, datetime, timedelta
from pathlib import Path

import ollama

MODEL = "qwen3:8b"

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

DATA_DIR = project_root / "data"
DATA_DIR.mkdir(exist_ok=True)

WORK_LOG = DATA_DIR / "work_log.jsonl"
LOG_FILE = WORK_LOG  # the file the tools read and write; Step 8 points it at a test file for a while

TODAY = date.today()

print("Log file:", LOG_FILE)
print("Today:   ", TODAY.isoformat())

Log file: B:\_Dev\Python\Agentic-Engineering-Worklog\data\work_log.jsonl
Today:    2026-11-16


### How to read it

The full path of the log file, which should end in `Agentic-Engineering-Worklog\data\work_log.jsonl`, and today's date.

### What it means

This is a new project, so this log starts empty. The `data` folder is in `.gitignore`, so your real work log never goes to GitHub.

`LOG_FILE` is the one name both tools use. In Step 8 it is pointed at a separate test file, so measuring the agent never adds fake entries to your real log.

## Step 2: Two tools as plain Python

`add_entry` is the same as lesson 2. `search_entries` is new: it reads the log, keeps the entries that match, and adds up the hours.

In [2]:
def read_entries():
    """Every entry in the current log file, oldest first."""
    if not LOG_FILE.exists():
        return []
    lines = LOG_FILE.read_text(encoding="utf-8").splitlines()
    return [json.loads(line) for line in lines if line.strip()]


def add_entry(project, description, hours=None):
    """Save one work log entry and return a short confirmation."""
    entry = {
        "logged_at": datetime.now().isoformat(timespec="seconds"),
        "project": project,
        "description": description,
        "hours": hours,
    }
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")
    return f"Saved entry for {project}."


def search_entries(project=None, date_from=None, date_to=None):
    """Find entries by project and date range (YYYY-MM-DD, both ends included). Returns text with a total."""
    matches = []
    for entry in read_entries():
        day = entry["logged_at"][:10]
        if project and entry["project"].lower() != project.lower():
            continue
        if date_from and day < date_from:
            continue
        if date_to and day > date_to:
            continue
        matches.append(entry)

    if not matches:
        return "No matching entries."

    lines = []
    total = 0.0
    for entry in matches:
        hours = entry["hours"]
        if hours is not None:
            total += float(hours)
        shown = f"{hours} h" if hours is not None else "hours not given"
        lines.append(f"{entry['logged_at'][:10]} | {entry['project']} | {entry['description']} | {shown}")
    lines.append(f"{len(matches)} entries. Total hours where given: {total:g}")
    return "\n".join(lines)


print(add_entry("Agentic Engineering Worklog", "Tested both tools by hand, no model involved", 0.25))
print()
print(search_entries(project="Agentic Engineering Worklog", date_from=TODAY.isoformat()))

Saved entry for Agentic Engineering Worklog.

2026-11-16 | Agentic Engineering Worklog | Tested both tools by hand, no model involved | 0.25 h
1 entries. Total hours where given: 0.25


### How to read it

First the confirmation from `add_entry`, then what `search_entries` found for this project today: one line per entry (date, project, description, hours) and a last line with the count and total hours.

If you run the cell again, you will see one more entry each time. That is expected: `add_entry` appends.

### What it means

As in lesson 2, both tools work before the model is involved. Two choices in `search_entries` matter later:

- **The code adds up the hours, not the model.** Small models make arithmetic slips. Handing the model a finished total means it only has to repeat a number, not calculate one.
- **It returns plain text, not raw JSON.** The model reads this text to answer you, and short labelled lines are easier for it to read correctly.

## Step 3: Describe both tools to the model

The model sees only these descriptions. The new one tells it the date format and that both dates are included.

In [3]:
ADD_ENTRY_TOOL = {
    "type": "function",
    "function": {
        "name": "add_entry",
        "description": "Save one work log entry. Use this when the user describes work they did.",
        "parameters": {
            "type": "object",
            "properties": {
                "project": {"type": "string", "description": "Name of the project the work was for."},
                "description": {"type": "string", "description": "Short description of what was done."},
                "hours": {"type": "number", "description": "Hours spent. Leave this out if the user did not say."},
            },
            "required": ["project", "description"],
        },
    },
}

SEARCH_ENTRIES_TOOL = {
    "type": "function",
    "function": {
        "name": "search_entries",
        "description": (
            "Look up saved work log entries. Use this to answer questions about past work. "
            "Returns the matching entries and the total hours."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "project": {"type": "string", "description": "Only entries for this project. Leave out for all projects."},
                "date_from": {"type": "string", "description": "First day to include, as YYYY-MM-DD."},
                "date_to": {"type": "string", "description": "Last day to include, as YYYY-MM-DD."},
            },
            "required": [],
        },
    },
}

TOOLS = [ADD_ENTRY_TOOL, SEARCH_ENTRIES_TOOL]
AVAILABLE_TOOLS = {"add_entry": add_entry, "search_entries": search_entries}

KNOWN_PROJECTS = [
    "Cornerpost Watch", "Cornerpost Logic", "Herd Planner", "Agentic Engineering Worklog",
]

SYSTEM_PROMPT = (
    "You are a work log assistant. "
    f"Today is {TODAY.isoformat()} ({TODAY.strftime('%A')}). "
    "When the user describes work they did, call add_entry. "
    "When the user asks about past work, call search_entries and answer from its result. "
    "If a request needs both, save first and search after. "
    "Known projects: " + ", ".join(KNOWN_PROJECTS) + ". "
    "Use one of those names when the work matches one. "
    "Only include hours if the user stated them. Convert minutes to hours. "
    "If the message is not about work, answer normally without calling a tool."
)


def call_model(messages):
    """One call to the local model with both tools on offer."""
    return ollama.chat(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        think=False,
        options={"temperature": 0},
    )


print(SYSTEM_PROMPT)

You are a work log assistant. Today is 2026-11-16 (Monday). When the user describes work they did, call add_entry. When the user asks about past work, call search_entries and answer from its result. If a request needs both, save first and search after. Known projects: Cornerpost Watch, Cornerpost Logic, Herd Planner, Agentic Engineering Worklog. Use one of those names when the work matches one. Only include hours if the user stated them. Convert minutes to hours. If the message is not about work, answer normally without calling a tool.


### How to read it

The system prompt the model will get, with today's date filled in.

### What it means

Three things are new compared with lesson 2:

- **Today's date is in the prompt.** Without it, the model has no idea what "today" or "yesterday" means and would guess a date.
- **"Save first and search after"** tells the model the order for two-step requests.
- **"Convert minutes to hours"** heads off "30 minutes" being saved as `30`.

Each line in the prompt fixes a mistake you would otherwise have to catch later. Writing these well is a large part of building agents.

## Step 4: Watch the lesson 2 program fail

This is lesson 2's `run_one_turn`, unchanged except that it now has both tools. It lets the model ask for tools once, runs them, and asks for a final answer.

In [4]:
TWO_STEP_REQUEST = (
    "Log 1 hour on Herd Planner updating the calving chart, "
    "then tell me how many hours I've logged on Herd Planner today."
)


def run_one_turn(user_text):
    """Lesson 2's approach: one round of tool requests, then a final reply."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    response = call_model(messages)
    if not response.message.tool_calls:
        return response.message.content

    messages.append(response.message)
    for call in response.message.tool_calls:
        name = call.function.name
        arguments = dict(call.function.arguments)
        result = AVAILABLE_TOOLS[name](**arguments)
        print(f"  [tool] {name}({arguments})")
        messages.append({"role": "tool", "tool_name": name, "content": result})

    final = call_model(messages)
    if final.message.tool_calls:
        wanted = [c.function.name for c in final.message.tool_calls]
        print(f"  The model asked for more tools ({', '.join(wanted)}), but this program has no second round.")
    return final.message.content


print("You:  ", TWO_STEP_REQUEST)
reply = run_one_turn(TWO_STEP_REQUEST)
print("Model:", repr(reply))

You:   Log 1 hour on Herd Planner updating the calving chart, then tell me how many hours I've logged on Herd Planner today.
  [tool] add_entry({'description': 'Updating the calving chart', 'hours': 1, 'project': 'Herd Planner'})
  [tool] search_entries({'date_from': '2026-11-16', 'date_to': '2026-11-16', 'project': 'Herd Planner'})
Model: 'You have logged 1 hour on Herd Planner today.'


### How to read it

Look at the `[tool]` lines, then the final reply. There are four things this can show:

- **One tool, then a note that the model wanted more.** It saved the entry, then asked to search, and nobody ran the search. The reply is usually empty (`''`).
- **Both tools in one round, save first.** The code runs requests in the order listed, so the total includes the new hour and the answer is right.
- **Both tools in one round, search first.** The search runs before the save, so the total is missing the new hour.
- **One tool and a confident answer.** It saved the entry and then stated a total without searching.

### What it means

The program decides how many rounds of tools there are, not the model. That is fine when the model can plan every step from the start, and it breaks when a step depends on what an earlier one returned.

**In my run** the model asked for both tools in one round, save first, so even this one-round program got the right total (1 hour at that point). The request did not actually need two rounds, because nothing about the search depended on the save's result. Exercise 4 uses a request that does.

This cell really did save one entry to your log. That is fine: it is real work you will see in later searches.

## Step 5: The loop

This is the whole idea of an agent: keep calling the model, run whatever tools it asks for, send back the results, and stop when it answers without asking for anything.

In [5]:
MAX_STEPS = 6


def run_agent(user_text, verbose=True):
    """Let the model call tools as many times as it needs, up to MAX_STEPS. Returns (reply, tool calls made)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    calls_made = []

    for step in range(1, MAX_STEPS + 1):
        response = call_model(messages)
        tool_calls = response.message.tool_calls or []

        if not tool_calls:
            if verbose:
                print(f"  step {step}: answered")
            return response.message.content, calls_made

        messages.append(response.message)
        for call in tool_calls:
            name = call.function.name
            arguments = dict(call.function.arguments)
            if name not in AVAILABLE_TOOLS:
                result = f"Error: there is no tool named {name}."
            else:
                try:
                    result = AVAILABLE_TOOLS[name](**arguments)
                except TypeError as error:
                    result = f"Error: {error}"
            calls_made.append((name, arguments))
            if verbose:
                first_line = result.splitlines()[0] if result else ""
                print(f"  step {step}: {name}({arguments}) -> {first_line}")
            messages.append({"role": "tool", "tool_name": name, "content": result})

    return f"Stopped after {MAX_STEPS} steps without a final answer.", calls_made


print("Loop ready. Step limit:", MAX_STEPS)

Loop ready. Step limit: 6


### How to read it

One line confirming the loop is defined. Nothing has been sent to the model yet.

### What it means

Compare `run_agent` with `run_one_turn` above. The only real change is the `for step in range(...)` loop around the call. That small change moves the decision about "what next?" from your code to the model.

Two safety features matter:

- **`MAX_STEPS`.** If the model keeps asking for tools, for example searching again and again without settling on an answer, the loop stops after 6 rounds instead of running forever. Every real agent needs a limit like this.
- **Errors go back to the model as text.** A wrong tool name or a bad argument doesn't crash the program. The model reads the error and can try again on the next step.

## Step 6: Run the agent on the two-step request

In [6]:
print("You:  ", TWO_STEP_REQUEST)
reply, calls = run_agent(TWO_STEP_REQUEST)
print("Model:", reply)

You:   Log 1 hour on Herd Planner updating the calving chart, then tell me how many hours I've logged on Herd Planner today.
  step 1: add_entry({'description': 'Updating the calving chart', 'hours': 1, 'project': 'Herd Planner'}) -> Saved entry for Herd Planner.
  step 1: search_entries({'date_to': '2026-11-16', 'project': 'Herd Planner', 'date_from': '2026-11-16'}) -> 2026-11-16 | Herd Planner | Updating the calving chart | 1 h
  step 2: answered
Model: You have logged 2 hours on Herd Planner today.


### How to read it

One line per tool call, labelled with the step it happened in, then "answered". Two good results are possible:

- **Two separate steps:** `add_entry` in step 1, `search_entries` in step 2, then the answer in step 3.
- **Both in step 1:** the model asks for both at once, save listed first, then answers in step 2.

Either way, the total should include the new hour. It also counts the hour saved in Step 4, and the hour from each time you've run these cells. To check it, open `data\work_log.jsonl` in PyCharm and add up today's Herd Planner hours yourself.

### What it means

**In my run** both calls came in step 1, and the answer (2 hours) was right. The model planned the whole job up front, which is efficient when no step depends on another's result.

What the loop adds is the *option* of another round. When the model does need to see a result before deciding, the loop gives it that chance, and the one-round program does not. Exercise 4 shows a request where that matters.

## Step 7: Questions that only need a search

The same loop handles questions about past work, and plain questions with no tool at all.

In [7]:
for text in [
    "How many hours have I logged on Herd Planner today?",
    "What have I worked on today?",
    "How many days are in a leap year?",
]:
    print("You:  ", text)
    reply, calls = run_agent(text)
    print("Model:", reply)
    print()

You:   How many hours have I logged on Herd Planner today?
  step 1: search_entries({'date_from': '2026-11-16', 'date_to': '2026-11-16', 'project': 'Herd Planner'}) -> 2026-11-16 | Herd Planner | Updating the calving chart | 1 h
  step 2: answered
Model: You have logged 2 hours on Herd Planner today.

You:   What have I worked on today?
  step 1: search_entries({'date_from': '2026-11-16', 'date_to': '2026-11-16'}) -> 2026-11-16 | Agentic Engineering Worklog | Tested both tools by hand, no model involved | 0.25 h
  step 2: answered
Model: Today, November 16, 2026, you worked on the following:

1. **Agentic Engineering Worklog**: Tested both tools by hand, no model involved (0.25 hours).
2. **Herd Planner**: Updating the calving chart (1 hour).
3. **Herd Planner**: Updating the calving chart (1 hour).

Total hours worked today: 2.25 hours.

You:   How many days are in a leap year?
  step 1: answered
Model: A leap year has 366 days.



### How to read it

- The first two should each show one `search_entries` step, then an answer built from what it found.
- The third should go straight to "answered" with no tool steps.

### What it means

One program now handles three kinds of request: saving, looking up, and plain questions. It also handles combinations of them. You didn't write a branch for each kind; the model chooses on every step.

## Step 8: Measure it

A test set with a known starting log, so the right answers are known in advance. Each test starts from the same four entries in a separate test file, so your real log is never touched.

Each case is scored three ways:

- **Tools:** did it use the right tools (and no others)?
- **Answer:** does the reply contain the facts it should?
- **Saved:** were the right entries saved, with the right project and hours?

In [8]:
TEST_LOG = DATA_DIR / "test_log_lesson3.jsonl"
YESTERDAY = (TODAY - timedelta(days=1)).isoformat()
TODAY_S = TODAY.isoformat()

STARTING_ENTRIES = [
    {"logged_at": f"{YESTERDAY}T09:00:00", "project": "Cornerpost Watch", "description": "Tuned the cloud mask", "hours": 1},
    {"logged_at": f"{YESTERDAY}T14:00:00", "project": "Agentic Engineering Worklog", "description": "Ran lesson 2", "hours": None},
    {"logged_at": f"{TODAY_S}T08:30:00", "project": "Cornerpost Watch", "description": "Fixed the alert email bug", "hours": 2},
    {"logged_at": f"{TODAY_S}T10:00:00", "project": "Herd Planner", "description": "Added vaccine records", "hours": 1.5},
]

TEST_CASES = [
    {"text": "Log 1 hour on Herd Planner updating the calving chart, then tell me how many hours I've logged on Herd Planner today.",
     "tools": {"add_entry", "search_entries"}, "answer": [r"\b2\.5\b"],
     "saved": [("Herd Planner", 1)]},
    {"text": "How many hours have I spent on Cornerpost Watch in total?",
     "tools": {"search_entries"}, "answer": [r"\b3(\.0)?\b"], "saved": []},
    {"text": "What did I work on yesterday?",
     "tools": {"search_entries"}, "answer": [r"cloud mask", r"lesson 2"], "saved": []},
    {"text": "Spent 30 minutes on the Cornerpost Logic company site.",
     "tools": {"add_entry"}, "answer": [], "saved": [("Cornerpost Logic", 0.5)]},
    {"text": "Log two things: 2 hours on Cornerpost Watch writing alert tests, and 1 hour on Herd Planner fixing the weight chart.",
     "tools": {"add_entry"}, "answer": [], "saved": [("Cornerpost Watch", 2), ("Herd Planner", 1)]},
    {"text": "What is the capital of Colorado?",
     "tools": set(), "answer": [r"denver"], "saved": []},
]


def reset_test_log():
    TEST_LOG.write_text("".join(json.dumps(e) + "\n" for e in STARTING_ENTRIES), encoding="utf-8")


def same_hours(got, expected):
    try:
        return float(got) == float(expected)
    except (TypeError, ValueError):
        return False


scores = {"tools": 0, "answer": 0, "saved": 0}
LOG_FILE = TEST_LOG
try:
    for case in TEST_CASES:
        reset_test_log()
        reply, calls = run_agent(case["text"], verbose=False)
        new_entries = read_entries()[len(STARTING_ENTRIES):]

        tools_ok = {name for name, _ in calls} == case["tools"]
        answer_ok = all(re.search(p, reply or "", re.IGNORECASE) for p in case["answer"])
        got_saved = sorted((e["project"], e["hours"]) for e in new_entries)
        want_saved = sorted(case["saved"])
        saved_ok = len(got_saved) == len(want_saved) and all(
            g[0] == w[0] and same_hours(g[1], w[1]) for g, w in zip(got_saved, want_saved)
        )

        scores["tools"] += tools_ok
        scores["answer"] += answer_ok
        scores["saved"] += saved_ok

        print(case["text"])
        print(f"   tools: {'right' if tools_ok else 'WRONG'} {[name for name, _ in calls]}")
        print(f"   answer: {'right' if answer_ok else 'WRONG'}   saved: {'right' if saved_ok else 'WRONG'} {got_saved}")
        print(f"   reply: {(reply or '').strip()[:160]}")
        print()
finally:
    LOG_FILE = WORK_LOG

total = len(TEST_CASES)
print(f"Right tools: {scores['tools']} of {total}")
print(f"Right answer: {scores['answer']} of {total}")
print(f"Right saves: {scores['saved']} of {total}")
print()
print("Log file is back to:", LOG_FILE.name)

Log 1 hour on Herd Planner updating the calving chart, then tell me how many hours I've logged on Herd Planner today.
   tools: right ['add_entry', 'search_entries']
   answer: right   saved: right [('Herd Planner', 1)]
   reply: You have logged 2.5 hours on Herd Planner today.

How many hours have I spent on Cornerpost Watch in total?
   tools: right ['search_entries']
   answer: right   saved: right []
   reply: You have spent a total of 3 hours on Cornerpost Watch.

What did I work on yesterday?
   tools: right ['search_entries']
   answer: right   saved: right []
   reply: Yesterday, you worked on the following:

1. **Cornerpost Watch**: Tuned the cloud mask (1 hour)
2. **Agentic Engineering Worklog**: Ran lesson 2 (hours not spec

Spent 30 minutes on the Cornerpost Logic company site.
   tools: right ['add_entry']
   answer: right   saved: right [('Cornerpost Logic', 0.5)]
   reply: The entry for the Cornerpost Logic company site has been saved. Let me know if you need anything el

### How to read it

For each test: the request, then whether the tools, the answer and the saved entries were right, with what actually happened in brackets, and the start of the reply. At the bottom, three scores out of 6, and a line confirming the tools point at your real log again.

What the right answers are, from the four starting entries:

- Herd Planner today was 1.5 hours, so after logging 1 more the answer should be **2.5**.
- Cornerpost Watch in total is 1 + 2 = **3** hours.
- Yesterday's work was **the cloud mask** and **lesson 2**.
- 30 minutes should be saved as **0.5** hours.
- The two-item request should save **two** entries.

### What it means

This test set checks things lesson 2's could not: the order of steps, whether a search is used instead of a guess, and whether a request with two items saves two entries.

A WRONG is useful, not a failure. It tells you what to fix next: the tool description, the system prompt, or the tool's own code. Change one thing at a time, and run this cell again to see if the score moves.

The answer checks are deliberately simple pattern matches. A reply could be correct but worded so the pattern misses it, so read any WRONG answer before deciding the agent got it wrong.

## Exercises

1. **Lower the step limit.** Set `MAX_STEPS = 1` in Step 5 and run Step 6 again. What does the agent do with a two-step request now? Set it back to 6 afterwards.
2. **Remove "save first and search after"** from the system prompt in Step 3, re-run Steps 3, 5 and 8, and compare the scores.
3. **Add a test case** for a project that is not in the list, such as "Spent an hour on my portfolio site." What project name does the model choose?
4. **A request where the second step depends on the first.** Run this in a new cell:

    ```python
    reply, calls = run_agent(
        "How many hours have I logged on Herd Planner today? "
        "If it's less than 5, log 1 more hour of chart cleanup."
    )
    print(reply)
    ```

    The model cannot know whether to log until it has seen the search result, so it should search in one step and decide in the next. Then try the same sentence with `run_one_turn` from Step 4 and compare.

## What it all means

- An agent is the loop: the model sees each tool's result before choosing the next step.
- The code still decides *what the model can do* (the tools) and *how long it may try* (the step limit). The model decides *what to do next*.
- Doing exact work in code (sums, dates, saving) and leaving the language to the model makes the agent more reliable.
- A test set with a known starting state turns "it seems to work" into scores you can compare after every change.

## What's next and why

**Lesson 4: Memory.** Right now every request starts fresh. If you say "log an hour on Herd Planner", then "actually make that two", the agent has no idea what "that" means. Lesson 4 keeps the conversation between requests and adds a way to correct the last entry, so the agent can follow a back-and-forth the way a person would.